# Content Safety Classifier using DeBERTa-v3

This notebook trains a binary content safety classifier using:
- **Model**: microsoft/deberta-v3-base
- **Dataset**: nvidia/Aegis-AI-Content-Safety-Dataset-2.0
- **Task**: Binary classification (Safe vs Unsafe) on prompts only

## 1. Install Dependencies

In [ ]:
!pip install datasets scikit-learn scipy tqdm accelerate -q

## 2. Imports

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    DebertaV2ForSequenceClassification,
    DebertaV2Tokenizer,
    DebertaV2Config,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from huggingface_hub import hf_hub_download
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
from scipy.special import softmax
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"
DATASET_NAME = "nvidia/Aegis-AI-Content-Safety-Dataset-2.0"
OUTPUT_DIR = "./content_safety_model"

NUM_LABELS = 2
NUM_EPOCHS = 3
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max length: {MAX_LENGTH}")

## 4. Load NVIDIA Aegis Dataset

In [ ]:
print(f"Loading {DATASET_NAME}...")

# Load all splits
dataset = load_dataset(DATASET_NAME)

print(f"\nDataset splits:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} samples")

print(f"\nColumns: {dataset['train'].column_names}")

## 5. Prepare Dataset (Prompts Only)

In [ ]:
def prepare_prompt_dataset(dataset_split):
    """
    Extract prompts and their labels from the dataset.
    Focuses only on prompt safety, ignoring response labels.
    """
    texts = []
    labels = []
    categories = []
    
    for sample in tqdm(dataset_split, desc="Processing"):
        prompt = sample.get('prompt', '')
        prompt_label = sample.get('prompt_label', None)
        violated_cats = sample.get('violated_categories', '')
        
        # Skip if prompt is redacted or empty
        if not prompt or prompt == 'REDACTED' or prompt_label is None:
            continue
        
        # Convert label: 'safe' -> 0, 'unsafe' -> 1
        if isinstance(prompt_label, str):
            label = 0 if prompt_label.lower() == 'safe' else 1
        else:
            label = int(prompt_label)
        
        texts.append(prompt)
        labels.append(label)
        categories.append(violated_cats if violated_cats else 'safe')
    
    return texts, labels, categories


print("Processing training data...")
train_texts, train_labels, train_cats = prepare_prompt_dataset(dataset['train'])

print("\nProcessing validation data...")
val_texts, val_labels, val_cats = prepare_prompt_dataset(dataset['validation'])

print("\nProcessing test data...")
test_texts, test_labels, test_cats = prepare_prompt_dataset(dataset['test'])

# Statistics
def print_stats(name, labels):
    safe = sum(1 for l in labels if l == 0)
    unsafe = sum(1 for l in labels if l == 1)
    print(f"  {name}: {len(labels)} samples (Safe: {safe}, Unsafe: {unsafe})")

print("\nDataset Statistics:")
print_stats("Train", train_labels)
print_stats("Validation", val_labels)
print_stats("Test", test_labels)

## 6. Create HuggingFace Datasets

In [ ]:
train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "label": val_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "label": test_labels
})

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")

## 7. Load DeBERTa-v3 Model with LayerNorm Fix

In [ ]:
def fix_deberta_state_dict(state_dict):
    """
    Fix DeBERTa-v3 LayerNorm naming convention.
    Renames .gamma/.beta to .weight/.bias for compatibility.
    """
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key
        
        # Fix LayerNorm naming
        if '.gamma' in key:
            new_key = key.replace('.gamma', '.weight')
        elif '.beta' in key:
            new_key = key.replace('.beta', '.bias')
        
        new_state_dict[new_key] = value
    
    return new_state_dict


print(f"\nLoading {MODEL_NAME}...")

# Load tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)

# Load config and create model
config = DebertaV2Config.from_pretrained(MODEL_NAME)
config.num_labels = NUM_LABELS
config.id2label = {0: "SAFE", 1: "UNSAFE"}
config.label2id = {"SAFE": 0, "UNSAFE": 1}

# Create model with random classifier weights
model = DebertaV2ForSequenceClassification(config)

# Download and load pretrained weights
print("Downloading pretrained weights...")
weights_path = hf_hub_download(repo_id=MODEL_NAME, filename="pytorch_model.bin")
pretrained_state_dict = torch.load(weights_path, map_location="cpu")

# Fix the LayerNorm naming
print("Fixing LayerNorm naming convention...")
fixed_state_dict = fix_deberta_state_dict(pretrained_state_dict)

# Load the fixed weights (strict=False to allow missing classifier weights)
missing, unexpected = model.load_state_dict(fixed_state_dict, strict=False)

print(f"\nWeight loading complete!")
print(f"  Missing keys (expected - classifier): {len(missing)}")
print(f"  Unexpected keys (MLM head - ignored): {len(unexpected)}")
print(f"\nModel loaded: {config.model_type}")
print(f"Parameters: {model.num_parameters():,}")

## 8. Tokenize Datasets

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print(f"Tokenization complete!")
print(f"  Train features: {tokenized_train.features}")

## 9. Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)
    preds = np.argmax(probs, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', pos_label=1
    )
    
    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        auc = 0.0
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc,
    }

print("Metrics function defined!")

## 10. Train the Model

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    report_to="none",
    seed=RANDOM_SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized!")

In [ ]:
print("\n" + "="*60)
print("  STARTING TRAINING")
print("="*60)

trainer.train()

print("\nTraining complete!")

## 11. Save the Model

In [ ]:
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 12. Evaluate on Test Set

In [ ]:
print("\n" + "="*60)
print("  EVALUATION ON TEST SET")
print("="*60)

# Get predictions
predictions = trainer.predict(tokenized_test)
logits = predictions.predictions
probs = softmax(logits, axis=1)
preds = np.argmax(probs, axis=1)
labels = np.array(test_labels)

# Calculate metrics
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, preds, average='binary', pos_label=1
)

try:
    auc = roc_auc_score(labels, probs[:, 1])
except ValueError:
    auc = 0.0

print(f"\nAccuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1 Score:  {f1*100:.2f}%")
print(f"AUC-ROC:   {auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(labels, preds)
print(f"\nConfusion Matrix:")
print(f"                  Predicted")
print(f"                  Safe    Unsafe")
print(f"  Actual Safe     {cm[0][0]:5d}    {cm[0][1]:5d}")
print(f"  Actual Unsafe   {cm[1][0]:5d}    {cm[1][1]:5d}")

# Classification Report
print(f"\nClassification Report:")
print(classification_report(labels, preds, target_names=['Safe', 'Unsafe']))

print(f"False Negatives (missed unsafe): {cm[1][0]}")
print(f"False Positives (blocked safe): {cm[0][1]}")

## 13. Test Example Prompts

In [ ]:
print("\n" + "="*60)
print("  EXAMPLE PREDICTIONS")
print("="*60)

examples = [
    "What is the capital of France?",
    "How do I make a homemade bomb?",
    "Can you help me write a resume?",
    "Tell me how to hack into someone's email account.",
    "What are some healthy dinner recipes?",
    "How can I hurt myself without anyone noticing?",
    "Explain quantum computing in simple terms.",
    "Write a story with extremely violent content.",
]

model.eval()
device = next(model.parameters()).device

for prompt in examples:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits.cpu().numpy(), axis=1)[0]
    
    safe_prob = probs[0] * 100
    unsafe_prob = probs[1] * 100
    verdict = "BLOCKED (Unsafe)" if probs[1] > 0.5 else "ALLOWED (Safe)"
    
    print(f'\nPrompt: "{prompt[:60]}..."' if len(prompt) > 60 else f'\nPrompt: "{prompt}"')
    print(f"  SAFE: {safe_prob:.1f}%  |  UNSAFE: {unsafe_prob:.1f}%")
    print(f"  Verdict: {verdict}")

## 14. Download Model

In [ ]:
import shutil

# Create zip file for download
shutil.make_archive('content_safety_model', 'zip', OUTPUT_DIR)
print("Model zipped as 'content_safety_model.zip'")
print("You can download it from the Output section on the right panel.")